# 4.6b Symptoms in the pregnancy window, and message length per topic

**Notebook 4.6 is split in four:** [a: move and Benji](roos_04.6a-move-and-baby.ipynb) (§4.6.1–4.6.9) · [b: symptoms and topics](roos_04.6b-symptoms-and-topics.ipynb) (§4.6.10–4.6.11) · [c: graded visualisation 3](roos_04.6c-graded-vis3-positive-test.ipynb) (§4.6.12) · [d: the hard weeks](roos_04.6d-hard-weeks.ipynb) (§4.6.13). Loading, the four dates, `save` and the helpers §4.6.13 reuses live in [roos_046_shared.py](roos_046_shared.py).

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.analytics import DistributionFitter, FitResult
from goad_toolkit.datatransforms import Pipeline, RegexFeature
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.visualizer import (
    HorizontalLine,
    PlotSettings,
)
from roos_046_shared import (
    BABY_DATE,
    PREGNANCY_START,
    load_own,
)

from scripts.plots import BarPlot
from scripts.roos_plots import DistributionDiffPlot, RidgelinePlot, ViolinPlot


## Loading your data

Same chat as 4.6a: `load_own()` (in `roos_046_shared.py`) loads the export, merges Louis's three names into one, and adds `n_chars`.

In [ ]:
own = load_own()
own.head()

## 4.6.10 Does the pregnancy window stand out? Nausea, fatigue, and both combined

Three questions, one recipe, reused three times:

1. **Nausea** — a comparison chart and a distribution chart, since that's what was
   asked for both ways.
2. **Fatigue** — the distribution chart alone.
3. **Nausea + fatigue combined** — same distribution chart, on "mentions either
   symptom," to answer directly: is the pregnancy window recognisable against the rest
   of the chat?

**Same categories as `roos_03.3`**, unchanged (`NAUSEA_PATTERN`, `FATIGUE_PATTERN`,
English/Dutch/French each). **Same pregnancy window**, unchanged (`PREGNANCY_START` =
`BABY_DATE` − 41 weeks, the last-period due-date convention from `roos_03.3`).

New here: instead of a weekly *line* (03.3's approach), each category becomes a
**weekly count**, fitted with `DistributionFitter` and split **during pregnancy** vs.
**the rest of the chat** — the same "messages per day" recipe goal 4.17 asks for, at
weekly resolution because nausea alone is rare enough (n=38 all-time) that daily counts
would be almost entirely zero either way.

Both chart types are `scripts/plots.py` classes: `BarPlot` for the comparison,
`DistributionDiffPlot` (4.6.9) for the distribution — reused as-is, not rebuilt.

In [ ]:
# Nausea/fatigue patterns unchanged from roos_03.3 (PREGNANCY_START = BABY_DATE - 41 weeks,
# in roos_046_shared.py).
NAUSEA_PATTERN = (
    r"(?i)("
    r"\bnausea(ted|ous)?\b|\bqueasy\b|"
    r"misselijk(heid)?|"
    r"naus[ée]e?(ux|se)?|mal au c[oœ]eur"
    r")"
)
FATIGUE_PATTERN = (
    r"(?i)("
    r"\bfatigue(d)?\b|\btired\b|\bexhaust(ed|ion)\b|\bworn out\b|"
    r"\bmoe\b|\bmoeheid\b|\bvermoeidheid\b|\bvermoeid\b|\buitgeput\b|"
    r"fatigu[ée]e?|[ée]puis[ée]e?|crev[ée]e?"
    r")"
)

symptoms = (
    Pipeline()
    .add(RegexFeature, name="nausea_words", column="message", pattern=NAUSEA_PATTERN,
         feature="n_nausea_words", mode="count")
    .add(RegexFeature, name="fatigue_words", column="message", pattern=FATIGUE_PATTERN,
         feature="n_fatigue_words", mode="count")
    .apply(own)
)
symptoms["nausea"] = symptoms["n_nausea_words"].gt(0)
symptoms["fatigue"] = symptoms["n_fatigue_words"].gt(0)
symptoms["either"] = symptoms["nausea"] | symptoms["fatigue"]
symptoms["timestamp_naive"] = symptoms["timestamp"].dt.tz_convert(None)

print(symptoms[["nausea", "fatigue", "either"]].sum().rename("messages mentioning it"))


def weekly_in_out(frame: pd.DataFrame, flag_col: str) -> pd.DataFrame:
    """Weekly count of messages where flag_col is True, reindexed to the chat's full
    weekly range (quiet weeks are zero, not missing), labelled by whether that week
    falls inside the pregnancy window.

    The full range comes from resampling the UNFILTERED frame with the same "W" rule
    (not a hand-built `pd.date_range(min, max, freq="W")` -- that keeps whichever
    time-of-day the first raw message happened to carry, which never lines up with
    `resample`'s own midnight-Sunday bins, and silently reindexes every real count to
    zero). `rename_axis("week")` before `reset_index()` names the resulting column
    directly, rather than relying on `reset_index()`'s own naming -- which follows the
    Series' index name (inherited from `set_index("timestamp_naive")` further up, not
    the generic "index"), and renaming the wrong column name is a silent no-op, not an
    error, which is exactly what let this bug through once already.
    """
    weekly = frame[frame[flag_col]].set_index("timestamp_naive").resample("W").size()
    full_range = frame.set_index("timestamp_naive").resample("W").size().index
    weekly = (
        weekly.reindex(full_range, fill_value=0)
        .rename("n")
        .rename_axis("week")
        .reset_index()
    )
    weekly["in_pregnancy"] = weekly["week"].between(PREGNANCY_START, BABY_DATE)
    return weekly


def fit_in_out(weekly: pd.DataFrame) -> dict:
    """Fit the discrete registry separately inside vs. outside the pregnancy window.
    Falls back to a plain poisson (always fittable, even on near-all-zero data) when
    the registry search finds nothing usable -- reported explicitly, not hidden."""
    fitter = DistributionFitter(DistributionRegistry(), seed=42)
    fits = {}
    for label, is_in in [("outside pregnancy", False), ("during pregnancy", True)]:
        values = weekly.loc[weekly["in_pregnancy"] == is_in, "n"].to_numpy().astype(float)
        candidates = fitter.fit(values, discrete=True)
        best = next(
            (f for f in candidates if isinstance(f, FitResult) and f.best_likelihood), None
        )
        if best is None:
            best = fitter.fit_distribution("poisson", values)
            print(f"{label:18s} registry fit found nothing usable -- fell back to poisson")
        fits[label] = (values, best)
        print(
            f"{label:18s} {best.distribution:8s} mean {best.frozen_dist.mean():5.2f}  "
            f"sd {best.frozen_dist.std():5.2f}  (n={len(values)} weeks)"
        )
    return fits


def plot_pregnancy_diff(fits: dict, title: str) -> tuple:
    """DistributionDiffPlot of P(during pregnancy) - P(outside), same recipe as 4.6.9."""
    out_values, out_fit = fits["outside pregnancy"]
    in_values, in_fit = fits["during pregnancy"]
    max_x = max(
        int(np.ceil(out_fit.frozen_dist.ppf(0.99))),
        int(np.ceil(in_fit.frozen_dist.ppf(0.99))),
        1,
    )
    x = np.arange(0, max_x + 1)
    diff = in_fit.frozen_dist.pmf(x) - out_fit.frozen_dist.pmf(x)

    settings = PlotSettings(
        figsize=(10, 5),
        title=title,
        xlabel="mentions per week",
        ylabel="P(during pregnancy) − P(outside pregnancy)",
    )
    host = DistributionDiffPlot(settings)
    fig, ax = host.plot(x=x, diff=diff, label="pregnancy − rest of chat",
                        color=settings.highlight_color)
    host.plot_on(HorizontalLine(settings), y=0)
    ax.legend()
    fig.text(
        0.5, -0.08,
        f"During pregnancy: {in_fit.distribution}, mean {in_fit.frozen_dist.mean():.2f}/week "
        f"(n={len(in_values)} weeks). Outside: {out_fit.distribution}, mean "
        f"{out_fit.frozen_dist.mean():.2f}/week (n={len(out_values)} weeks).",
        ha="center", fontsize=8.5, color="dimgray",
    )
    return fig, ax

### Nausea — comparison, then distribution

In [ ]:
nausea_weekly = weekly_in_out(symptoms, "nausea")
nausea_fits = fit_in_out(nausea_weekly)

nausea_summary = (
    nausea_weekly.groupby("in_pregnancy")["n"].mean().rename("mean_per_week").reset_index()
)
nausea_summary["period"] = nausea_summary["in_pregnancy"].map(
    {True: "during pregnancy", False: "outside pregnancy"}
)

nausea_bar_settings = PlotSettings(
    figsize=(7, 4.5),
    title="Nausea mentions: weekly average, in vs. outside the pregnancy window",
    xlabel="",
    ylabel="mean mentions per week",
)
fig, ax = BarPlot(nausea_bar_settings).plot(
    data=nausea_summary, x="period", y="mean_per_week",
    order=["outside pregnancy", "during pregnancy"],
    palette={
        "outside pregnancy": nausea_bar_settings.base_color,
        "during pregnancy": nausea_bar_settings.highlight_color,
    },
)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

In [ ]:
plot_pregnancy_diff(nausea_fits, "Nausea: does the pregnancy window stand out?")

### Fatigue — distribution only

In [ ]:
fatigue_weekly = weekly_in_out(symptoms, "fatigue")
fatigue_fits = fit_in_out(fatigue_weekly)
plot_pregnancy_diff(fatigue_fits, "Fatigue: does the pregnancy window stand out?")

### Nausea + fatigue combined — is pregnancy recognisable?

In [ ]:
either_weekly = weekly_in_out(symptoms, "either")
either_fits = fit_in_out(either_weekly)
plot_pregnancy_diff(either_fits, "Nausea + fatigue combined: is pregnancy recognisable?")

## 4.6.11 Message length per topic: ridgeline and violin

**Hypothesis, in your words:** messages about `planning_logistics` are longer than
messages from the other categories.

**Same categories as `roos_01.4`**, patterns copied over unchanged: love, financial,
child, climbing, running, food, planning_logistics. Two things to know before reading
the charts:

- **`food` is commented out in `roos_01.4`**, with no reason logged in
  `analysis-log.md`. It's back in here because you asked for it. If you dropped it on
  purpose (e.g. too many false positives from `courses`/`pasta`/`eggs`), drop it here too.
- **`be_careful` is left out**, same as you already did in 01.4's small multiples, and
  it wasn't on this list anyway.

Three choices, mine, flagged so you can overrule them:

- **One message can count in several categories.** "Can you pay the daycare when you're
  home?" counts under `financial`, `child` *and* `planning_logistics`. About 13% of
  categorised messages hit more than one. The alternative (one label per message) needs
  a tie-break rule you'd have to invent and defend.
- **An `all messages` reference row, in grey at the bottom.** Without it you can't tell
  "planning messages are long" from "*any* message that happens to contain a keyword is
  long". Delete it if you disagree, but read the tip at the end first.
- **Log x-axis.** 4.6.6 already showed message length has a long right tail. On a
  linear axis every ridge piles up against zero and the differences disappear. The
  black tick in each ridge is that group's median.

`planning_logistics` is the one in colour, because it's your prediction (grey-first,
same rule as 02.2). The rows are sorted by median, so the order itself answers
"which topic is longest".

In [ ]:
# Patterns copied unchanged from roos_01.4 (food un-commented, be_careful left out).
TOPIC_PATTERNS: dict[str, str] = {
    "love": (
        r"(?i)("
        r"je\s*t'?aime|tu me manques|mon amour|mon c[oœ]ur|ma belle|"
        r"mon ch[ée]ri|ma ch[ée]rie|je pense [aà] toi|bisous?|t'es (belle|beau)|"
        r"ik houd? van (je|jou)|ik mis (je|jou)|schatje|liefje|"
        r"i love you|i miss you|my love|sweetheart|darling|"
        r"❤️|😘|🥰|😍|💕|💖|💗|💓|💞|😻|💋"
        r"|je pense a toi|ik denk aan (je|jou)"
        r")"
    ),
    "financial": (
        r"(?i)("
        r"hypotheek(adviseur)?|hypoth[eè]que|mortgage( advisor)?|"
        r"rekening|compte( ?[eé]pargne)?|bank account|"
        r"belasting|imp[oô]ts?|\btaxe?s?\b|salaris|salaire|\bsalary\b|\bwage\b|"
        r"loon|huur|loyer|\brent\b|borg|caution|deposit|"
        r"spaarrekening|savings account|\bgeld\b|argent|\bmoney\b|"
        r"betalen|payer|\bpay\b|overmaken|virer|virement|transfer|"
        r"bunq|asn|\bing\b|rabobank|funda|velduil|"
        r"notaris|notaire|\bnotary\b|makelaar|agent immobilier|real estate agent|"
        r"iban|factuur|facture|invoice|budget"
        r")"
    ),
    "child": (
        r"(?i)("
        r"benji|b[ée]b[ée]|\bbaby\b|cr[eè]che|kinderdagverblijf|opvang|daycare|partou|basker|kinderdagopvang|"
        r"r[eé]cuperer|r[eé]cupes|prendre soin de|"
        r"\benfant\b|biberon|bottle[- ]?feeding|sucette|fopspeen|speentje|"
        r"pacifier|dummy|allaiter|donner les? seins?|borstvoeding|breastfeeding|t[ée]t[ée]e|"
        r"luiers?|diapers?|couches?|doekje|hydrophile|vitamines|hydrofiele doek|\bwipe\b|"
        r"kinderwagen|poussette|stroller|pram|la remorque|bike trailer|"
        r"consultatiebureau|kraamzorg|zwangerschap|grossesse|pregnan(cy|t)|"
        r"verloskundige|sage-femme|midwife|kolven|[ée]chographie"
        r")"
    ),
    "climbing": (
        r"(?i)("
        r"escalad(er|e)|salle de grimpe|grimpe|grimper|\bclimb(ing)?\b|climbing gym|"
        r"klimmen|klimmuur( centraal)?|\bboulder\b|\bbloc\b|\bvoie\b|"
        r"arkose|mountain network|\bncc\b|neoliet|\b[4-8][abc]\+?\b"
        r")"
    ),
    "running": (
        r"(?i)("
        r"\brun(ning)?\b|courir|\btrail\b|\bbornes?\b|kilom[eè]tre|\bkm\b|marathon|"
        r"hardlopen|hardloopschoenen|joggen|strava|parkrun|\b10k\b|\btempo\b|"
        r"chaussures|\bshoes\b"
        r")"
    ),
    "food": (
        r"(?i)("
        r"\beten\b|manger|\bd[iî]ner\b|avondeten|d[eé]jeuner|petit-d[eé]jeuner|\blunch\b|"
        r"ontbijt|breakfast|\bcook(ing)?\b|cuisiner|koken|recette|recept|recipe|"
        r"restaurant|\bhonger\b|\bfaim\b|hungry|dalle|"
        r"l[eé]gumes?|carottes?|courgettes?|poireaux?|oignons?|tomates?|pommes? de terre|"
        r"\bsalade\b|[ée]pinards?|brocolis?|poivrons?|champignons?|choux?|aubergines?|haricots?|"
        r"groenten|wortel(en)?|\bprei\b|uien?|tomaten?|aardappel(en)?|\bsla\b|spinazie|"
        r"paprika('s)?|paddenstoel(en)?|\bkool\b|bonen|"
        r"vegetables?|carrots?|zucchini|leeks?|onions?|tomatoe?s?|potatoe?s?|lettuce|spinach|"
        r"broccoli|peppers?|mushrooms?|cabbage|eggplant|beans?|"
        r"p[aâ]tes|pasta|œufs?|oeufs?|eggs?|\blait\b|\bmilk\b|melk|"
        r"fromage blanc|kwark|chocolat|chocolade|chocolate"
        r")"
    ),
    "planning_logistics": (
        r"(?i)("
        r"\bquand\b|je suis l[aà]|j'arrive|arriver|tu rentres quand|je rentre|en route|je pars|"
        r"[aà] quelle heure|hoe laat|\bwhen\b|i'll be there|\bthuis\b|\bhome\b|t'es partie?|"
        r"onderweg|on the way|\btrein\b|\btrain\b|vlucht|\bflight\b|\bagenda\b|"
        r"rendez-vous|afspraak|appointment|"
        r"boodschappen|courses?|lidl"
        r")"
    ),
}
ALL_LABEL = "all messages"

topic_pipeline = Pipeline()
for topic, pattern in TOPIC_PATTERNS.items():
    topic_pipeline.add(
        RegexFeature, name=f"{topic}_topic", column="message", pattern=pattern,
        feature=f"n_{topic}_topic", mode="count",
    )
with_topics = topic_pipeline.apply(own)

# Long format: one row per (message, topic it matches), so a message that hits two
# topics appears twice -- plus every message once more under ALL_LABEL.
topic_lengths = pd.concat(
    [
        with_topics.loc[with_topics[f"n_{topic}_topic"].gt(0), ["n_chars"]].assign(topic=topic)
        for topic in TOPIC_PATTERNS
    ]
    + [with_topics[["n_chars"]].assign(topic=ALL_LABEL)],
    ignore_index=True,
)

length_summary = (
    topic_lengths.groupby("topic")["n_chars"]
    .agg(n="size", median="median", mean="mean", p90=lambda s: s.quantile(0.9))
    .sort_values("median", ascending=False)
)
# Topics sorted by median, longest on top; the reference row always last.
topic_order = [t for t in length_summary.index if t != ALL_LABEL] + [ALL_LABEL]
topic_labels = {t: f"{t} (n={length_summary.loc[t, 'n']:,})" for t in topic_order}
length_summary.loc[topic_order].round(1)

In [ ]:
ridge_settings = PlotSettings(
    figsize=(10, 7),
    title="Message length per topic (black tick = median)",
    xlabel="characters per message (log scale)",
    ylabel="",
)
fig, ax = RidgelinePlot(ridge_settings).plot(
    data=topic_lengths, x="n_chars", group="topic",
    order=topic_order, labels=topic_labels, highlight_groups=["planning_logistics"],
)
fig.text(
    0.5, -0.04,
    "Each ridge scaled to the same height: compares shape, not how many messages a topic has. "
    "A message matching two topics is counted in both.",
    ha="center", fontsize=8.5, color="dimgray",
)

In [ ]:
violin_settings = PlotSettings(
    figsize=(10, 7),
    title="Message length per topic (dashed lines = quartiles, middle one = median)",
    xlabel="characters per message (log scale)",
    ylabel="",
)
fig, ax = ViolinPlot(violin_settings).plot(
    data=topic_lengths, x="n_chars", group="topic",
    order=topic_order, labels=topic_labels, highlight_groups=["planning_logistics"],
)
fig.text(
    0.5, -0.04,
    "Same data as the ridgeline. Each violin has the same maximum width, so width "
    "compares shape, not topic size.",
    ha="center", fontsize=8.5, color="dimgray",
)

Two charts, same data, two different questions:

- **Ridgeline**: where does each topic's *peak* sit, and does any topic have a
  second hump (e.g. a pile of one-emoji replies under `love`)?
- **Violin**: how do the medians and middle 50% line up? The quartile lines make
  "longer" a comparison you can read off the chart, not just an impression.

**My prediction, before looking:** `planning_logistics` messages are longer than the
other categories.

**What I found:** (write it here: where does `planning_logistics` actually rank, and
how far is *every* topic from the grey `all messages` row?)